In [1]:
import pandas as pd
import numpy as np
import re
import os
import joblib
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

In [2]:
df_train = pd.read_csv("../data/train.csv")
df_val   = pd.read_csv("../data/val.csv")
df_test  = pd.read_csv("../data/test.csv")

print("Train :", len(df_train))
print("Val :",   len(df_val))
print("Test :",  len(df_test))

Train : 4391
Val : 941
Test : 942


In [3]:
import sys
sys.path.append("..")
from src.preprocessing import clean_text, count_suspicious, has_base64

# Nettoyer les textes
df_train["text_clean"] = df_train["text"].apply(clean_text)
df_val["text_clean"]   = df_val["text"].apply(clean_text)
df_test["text_clean"]  = df_test["text"].apply(clean_text)

# Features manuelles
features_manuelles = ["text_length", "word_count",
                      "suspicious_count", "has_base64"]

for df in [df_train, df_val, df_test]:
    df["text_length"]      = df["text"].str.len()
    df["word_count"]       = df["text"].str.split().str.len()
    df["suspicious_count"] = df["text"].apply(count_suspicious)
    df["has_base64"]       = df["text"].apply(has_base64)

print("Preprocessing appliqué")

Preprocessing appliqué


In [4]:
# Charger le vectoriseur déjà entraîné dans l'étape précédente
tfidf = joblib.load("../models/tfidf_vectorizer.pkl")

# Transformer les textes
X_train_tfidf = tfidf.transform(df_train["text_clean"])
X_val_tfidf   = tfidf.transform(df_val["text_clean"])
X_test_tfidf  = tfidf.transform(df_test["text_clean"])

# Combiner avec les features manuelles
X_train = hstack([X_train_tfidf,
                  csr_matrix(df_train[features_manuelles].values)])
X_val   = hstack([X_val_tfidf,
                  csr_matrix(df_val[features_manuelles].values)])
X_test  = hstack([X_test_tfidf,
                  csr_matrix(df_test[features_manuelles].values)])

# Labels
y_train = df_train["label"].values
y_val   = df_val["label"].values
y_test  = df_test["label"].values

print("Features prêtes")
print("Shape X_train :", X_train.shape)

Features prêtes
Shape X_train : (4391, 9462)


In [5]:
print("Logistic Regression")

lr = LogisticRegression(
    max_iter=1000,      # iterations max pour converger
    C=1.0,              # force de régularisation
    random_state=42
)

lr.fit(X_train, y_train)

# Évaluation sur validation
y_pred_lr = lr.predict(X_val)
print(classification_report(y_val, y_pred_lr,
      target_names=["Bénin", "Malicieux"]))

# Sauvegarder
joblib.dump(lr, "../models/logistic_regression.pkl")
print("✅ Modèle sauvegardé")

Logistic Regression
              precision    recall  f1-score   support

       Bénin       0.97      0.89      0.93       407
   Malicieux       0.92      0.98      0.95       534

    accuracy                           0.94       941
   macro avg       0.95      0.94      0.94       941
weighted avg       0.94      0.94      0.94       941

✅ Modèle sauvegardé


In [6]:
print("Random Forest")

rf = RandomForestClassifier(
    n_estimators=200,   # nombre d'arbres
    max_depth=20,       # profondeur max de chaque arbre
    random_state=42,
    n_jobs=-1           # utilise tous les coeurs CPU disponibles
)

rf.fit(X_train, y_train)

# Évaluation sur validation
y_pred_rf = rf.predict(X_val)
print(classification_report(y_val, y_pred_rf,
      target_names=["Bénin", "Malicieux"]))

# Sauvegarder
joblib.dump(rf, "../models/random_forest.pkl")
print("✅ Modèle sauvegardé")

Random Forest
              precision    recall  f1-score   support

       Bénin       0.99      0.85      0.92       407
   Malicieux       0.90      1.00      0.94       534

    accuracy                           0.93       941
   macro avg       0.95      0.92      0.93       941
weighted avg       0.94      0.93      0.93       941

✅ Modèle sauvegardé


In [7]:
print("LinearSVC")

svc = LinearSVC(
    C=1.0,
    max_iter=5000,      
    random_state=42,
    tol=1e-4            
)

svc.fit(X_train, y_train)

y_pred_svc = svc.predict(X_val)
print(classification_report(y_val, y_pred_svc,
      target_names=["Bénin", "Malicieux"]))

joblib.dump(svc, "../models/linear_svc.pkl")
print("✅ Modèle sauvegardé")

LinearSVC
              precision    recall  f1-score   support

       Bénin       0.98      0.91      0.95       407
   Malicieux       0.94      0.99      0.96       534

    accuracy                           0.95       941
   macro avg       0.96      0.95      0.95       941
weighted avg       0.96      0.95      0.95       941

✅ Modèle sauvegardé


C:\Prompt_Injection_Detection\venv\Lib\site-packages\sklearn\svm\_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


In [8]:
from sklearn.metrics import accuracy_score, f1_score

# Résultats de tous les modèles sur validation
modeles = {
    "Logistic Regression" : (lr,  y_pred_lr),
    "Random Forest"       : (rf,  y_pred_rf),
    "LinearSVC"           : (svc, y_pred_svc),
}

print("=" * 55)
print(f"{'Modèle':<25} {'Accuracy':>10} {'F1-Score':>10}")
print("=" * 55)

for nom, (modele, y_pred) in modeles.items():
    acc = accuracy_score(y_val, y_pred)
    f1  = f1_score(y_val, y_pred, average="weighted")
    print(f"{nom:<25} {acc:>10.4f} {f1:>10.4f}")

print("=" * 55)

Modèle                      Accuracy   F1-Score
Logistic Regression           0.9426     0.9422
Random Forest                 0.9341     0.9333
LinearSVC                     0.9543     0.9541
